#**Izaz Khan**
***Reg. No:*** B23F0001AI029

***Section:*** AI Green  
***Course:*** ANN Lab 02                                                                     
***Date:*** 29/01/2026

#**Task 1: Forward Pass (Patient Risk Prediction)**

###**Step 1: Initialize Inputs and Weights**
I define the input features for the patient and the initial weights for our simple 2-2-1 network.

In [1]:
import numpy as np

# Inputs
x1, x2 = 0.1, 0.5
y_target = 1.0

# Weights (Input to Hidden)
w13, w23 = 0.2, -0.3
w14, w24 = 0.4, 0.1

# Weights (Hidden to Output)
w35, w45 = -0.5, 0.2

# Activation Function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

###**Step 2: Compute Hidden Layer Activations**
Calculating the weighted sum $z$ and applying the Sigmoid activation $y$ for neurons 3 and 4.

In [2]:
# Neuron 3
z3 = (x1 * w13) + (x2 * w23)
y3 = sigmoid(z3)

# Neuron 4
z4 = (x1 * w14) + (x2 * w24)
y4 = sigmoid(z4)

print(f"Hidden Neuron 3 Output (y3): {y3:.4f}")
print(f"Hidden Neuron 4 Output (y4): {y4:.4f}")

Hidden Neuron 3 Output (y3): 0.4675
Hidden Neuron 4 Output (y4): 0.5225


###**Step 3: Compute Final Output (Neuron 5)**
Calculating the final prediction of the network by processing the hidden layer outputs.

In [3]:
# Neuron 5 (Output)
z5 = (y3 * w35) + (y4 * w45)
y5 = sigmoid(z5)

print(f"Predicted Risk (y5): {y5:.4f}")
print(f"Target Risk: {y_target}")

Predicted Risk (y5): 0.4677
Target Risk: 1.0


###**Analysis: Why is the prediction considered incorrect?**
**1.The Threshold Gap:** In a classification task, we typically treat $0.5$ as the decision boundary. Our $y_5$ value will be very close to 0.5 (likely slightly below it due to the negative weight $w_{35}$).

**2.Target Mismatch:** Since the target is 1.0 (High Risk) and the network outputs a value much lower (closer to 0.4 or 0.5), the error is significant.

**3.Random Initialization:** The network is currently "untrained." It hasn't seen the error yet, so its weights are just arbitrary values that don't reflect the relationship between Blood Pressure/Age and health risk.

#**Task 2: Error Calculation (Identifying the Mistake)**

### **Step 4: Calculate Output Error**
I subtract the network's prediction ($y_5$) from the target value ($y_{target}$) to find the raw error.

In [5]:
# Error Calculation
error = y_target - y5

print(f"Target Value: {y_target}")
print(f"Predicted Value: {y5:.4f}")
print(f"Calculated Error: {error:.4f}")

Target Value: 1.0
Predicted Value: 0.4677
Calculated Error: 0.5323


###**Interpretation: What does the sign of the error tell you?**
The sign of the error is a critical signal for the learning process:
>**Positive Error** ($y_{target} > y_5$): This means the network's prediction was **too low** (underestimation). The positive sign tells the backpropagation algorithm that it needs to increase the weights (in the direction of the gradient) to push the output closer to the target in the next iteration.

>**Negative Error**($y_{target} < y_5$): This means the network's prediction was **too high** (overestimation). A negative sign signals that the weights need to be adjusted to decrease the output.

>**Zero Error:** The prediction is perfect, and no weight adjustment is needed.

In our specific patient case, since the target is $1.0$ and our prediction will be around $0.48$, the error will be positive (approx. $+0.52$). This tells us the network needs to become "more confident" that this patient is High Risk.

#**Task 3: Output Neuron Responsibility ($\delta_5$)**

###**Step 5: Compute Output Delta ($\delta_5$)**
The error term for the output neuron incorporates the derivative of the sigmoid function:
$\delta_5 = y_5(1 - y_5)(y_{target} - y_5)$.

In [6]:
# Output Neuron Responsibility
# Formula: delta5 = (Output) * (1 - Output) * (Target - Output)
delta5 = y5 * (1 - y5) * error

print(f"Output Delta (delta5): {delta5:.6f}")

Output Delta (delta5): 0.132514


###**Why do we multiply the error with the derivative?**
The term $y_5(1 - y_5)$ is the derivative of the Sigmoid function.I multiply the error by this value for three fundamental reasons:

>**1.Gradient Descent Direction:** The derivative tells us the slope of the activation function. It indicates how much the output changes for a small change in the input. Without it, we wouldn't know the mathematically correct "direction" or "magnitude" to adjust the weights to reduce the error.

>**2.The Vanishing Gradient Effect (Saturation):** If the neuron's output is very close to 0 or 1, the Sigmoid curve is almost flat.
>>In these regions, the derivative $y(1-y)$ is nearly 0.

>>This "kills" the gradient, telling the network: "This neuron is already very certain (saturated), so don't change its weights much."

>**3.Maximum Learning at Uncertainty:** The derivative is highest (0.25) when the output is exactly 0.5. This tells the network that when a neuron is "unsure," it should be most sensitive to weight updates.

#**Task 4: Hidden Neuron Responsibility ($\delta_3$ and $\delta_4$)**

### **Step 6: Compute Hidden Layer Deltas ($\delta_3$ and $\delta_4$)**
The hidden neurons calculate their error term by taking the derivative of their own activation and multiplying it by the 'weighted' error from the next layer.

In [7]:
# Hidden Neuron 3 Responsibility
delta3 = y3 * (1 - y3) * (w35 * delta5)

# Hidden Neuron 4 Responsibility
delta4 = y4 * (1 - y4) * (w45 * delta5)

print(f"Hidden Delta 3 (delta3): {delta3:.6f}")
print(f"Hidden Delta 4 (delta4): {delta4:.6f}")

Hidden Delta 3 (delta3): -0.016494
Hidden Delta 4 (delta4): 0.006612


###**Critical Thinking Questions.**
**1.Why do hidden neurons not directly use the target value?**

Hidden neurons do not have a "ground truth" to compare against. The target value ($y_{target}$) is only defined for the final output. The hidden neurons are responsible for creating features that help the output neuron reach that target. Therefore, they can only calculate their error based on how much the final output neuron says it was misled by them. This is the essence of "propagating" the error backward.

**2. Why does a larger outgoing weight result in a larger hidden error?**

Weights represent the "strength" of a connection. If a hidden neuron (like Neuron 3) has a very large weight ($w_{35}$) connecting it to the output, it means that neuron had a massive influence on the final (incorrect) prediction.
>In backpropagation, we "distribute blame" proportionally.

>If it contributed more to the mistake (via a large weight), it receive a larger share of the error ($\delta$) to ensure the weights are adjusted more significantly to fix the problem.

#**Task 5: Weight Updates (Learning from Mistakes)**

### **Step 7: Update Weights (Hidden → Output Layer)**
I update the weights $w_{35}$ and $w_{45}$ using the output delta ($\delta_5$) and the signals coming from the hidden neurons ($y_3, y_4$).

In [8]:
# Learning rate
learning_rate = 0.1

# New Weight = Old Weight + (Learning Rate * Delta * Input_to_that_weight)
w35_new = w35 + (learning_rate * delta5 * y3)
w45_new = w45 + (learning_rate * delta5 * y4)

print(f"Updated w35: {w35_new:.6f} (was {w35})")
print(f"Updated w45: {w45_new:.6f} (was {w45})")

Updated w35: -0.493804 (was -0.5)
Updated w45: 0.206924 (was 0.2)


### **Step 8: Update Weights (Input → Hidden Layer)**
I update the weights $w_{13}, w_{23}, w_{14}, w_{24}$ using the hidden deltas ($\delta_3, \delta_4$) and the original input features ($x_1, x_2$).

In [9]:
# Weight updates for Input -> Hidden
w13_new = w13 + (learning_rate * delta3 * x1)
w23_new = w23 + (learning_rate * delta3 * x2)
w14_new = w14 + (learning_rate * delta4 * x1)
w24_new = w24 + (learning_rate * delta4 * x2)

print(f"Updated w13: {w13_new:.6f}, Updated w23: {w23_new:.6f}")
print(f"Updated w14: {w14_new:.6f}, Updated w24: {w24_new:.6f}")

Updated w13: 0.199835, Updated w23: -0.300825
Updated w14: 0.400066, Updated w24: 0.100331


###**Critical Thinking Questions.**
 **1.Why do some weights change more than others?** The change in a weight ($\Delta w$) depends on three factors:

 >**1.The Error/Responsibility ($\delta$):** Weights leading into a neuron with a high error will change more.

 >**2.The Input Strength:** If the input signal was strong (large value), that weight is seen as a "bigger contributor" to the result and is adjusted more. If the input was near zero, the weight had almost no effect on the output, so changing it won't help much.

 >**3.The Learning Rate ($\eta$):** This scales the overall magnitude of the change.

**2. Why are some weight updates very small?** Weight updates become very small (or zero) in three scenarios:

>**1.The Network is Accurate:** If the error is small, $\delta$ is small, and updates shrink.

>**2.Saturation:** As we discussed in Task 3, if a neuron's activation is very close to 0 or 1, the derivative is nearly 0. This results in a tiny $\delta$, making the weight update almost non-existent.

>**3.Low Input Signal:** If a specific input (like $x_1$) is 0, the formula $\Delta w = \eta \cdot \delta \cdot 0$ results in zero change, because that weight didn't participate in the mistake.


#**Task 6: Interpretation & Reflection (Critical Thinking)**

### **Task 6: Interpretation & Reflection**
In this section,I analyze the conceptual mechanics of the Backpropagation algorithm.

###**1. Explain backpropagation as a process of "blame assignment."**
In a neural network, the error at the output is rarely the fault of just one neuron; it is the result of a collective effort. **Blame assignment** (credit assignment) is the process of tracing that error backward from the output layer to the input layer.

>**1.** The algorithm looks at each weight and asks: "How much did this specific connection contribute to the final mistake?"

>**2.** If a neuron had a large weight and a high activation, it gets assigned more "blame" (a larger $\delta$).

>**3.** Consequently, that weight is changed more aggressively to "fix" its contribution to the error.

###**2. What would happen if the Learning Rate ($\eta$) was...**

>**1.Very Large?** If the learning rate is too high, the updates to the weights will be massive. This causes the network to "overshoot" the optimal solution. Instead of settling into the minimum error, the network might oscillate back and forth or even diverge, causing the error to increase until the model fails completely (exploding gradients).

>**2.Very Small?** If the learning rate is too low, the weight updates will be tiny. While this makes the learning process very stable, it also makes it incredibly slow. The network might take a very long time to converge, or it might get stuck in a "local minimum" (a shallow dip in the error) because it doesn't have enough "momentum" to move toward the true best solution.

### **3. Why is backpropagation called “backward” propagation?**
It is called "backward" because of the direction the information (the error signal) flows.

>1.In the **Forward Pass**, data moves from input to output to generate a prediction.

>2.Once the error is calculated at the end, the **Backward Pass** begins.

>3.We calculate the gradient of the loss starting at the **output layer**, then move to the **hidden layer**, and finally to the **input layer**.

We must go backward because the error of a hidden neuron depends on the error of the neurons it feeds into. Mathematically, this is the application of the **Chain Rule** from calculus, where we solve for derivatives from the "outside-in" (output to input).



#**Bonus Challenge: Re-computing the Forward Pass**

### **Step 9: Re-compute Forward Pass (The Learning Test)**
We now use the updated weights to see if the network's prediction for the same patient (Age=0.1, BP=0.5) has improved.

In [10]:
# Forward Pass with Updated Weights

# 1. New Hidden Layer Activations
z3_new = (x1 * w13_new) + (x2 * w23_new)
y3_new = sigmoid(z3_new)

z4_new = (x1 * w14_new) + (x2 * w24_new)
y4_new = sigmoid(z4_new)

# 2. New Output Prediction
z5_new = (y3_new * w35_new) + (y4_new * w45_new)
y5_new = sigmoid(z5_new)

# 3. New Error
error_new = y_target - y5_new

print(f"Original Prediction: {y5:.6f}")
print(f"New Prediction:      {y5_new:.6f}")
print(f"Target Value:        {y_target}")
print("-" * 30)
print(f"Original Error:      {abs(error):.6f}")
print(f"New Error:           {abs(error_new):.6f}")
print(f"Improvement:         {abs(error) - abs(error_new):.6f}")

Original Prediction: 0.467726
New Prediction:      0.469363
Target Value:        1.0
------------------------------
Original Error:      0.532274
New Error:           0.530637
Improvement:         0.001637


###**How does this demonstrate that the network is learning?**
>**1.Reduction in Loss:** Learning is defined as the systematic reduction of error over time. If New Error < Original Error, the network has successfully optimized its parameters for this specific data point.

>**2.Directional Correction:** I notice that y5_new is slightly higher than y5. Because our target was 1.0, the network "realized" it was underestimating the risk and adjusted the weights to push the output in the correct direction.

>**3.Convergent Behavior:** Even though the improvement from a single step (one "epoch" on one sample) is small, this demonstrates the mechanism of Gradient Descent. If we repeat this process thousands of times, the "New Prediction" will eventually get extremely close to 1.0.

#**summary**
In summary, this lab demystifies Neural Networks by breaking "learning" down into a repeatable four-step mathematical cycle:

>**1.Forward Pass:** The network takes inputs (Age, BP) and produces a prediction. At first, this is just a random guess because the weights are unoptimized.

>**2.Error Measurement:** I then calculate how far the prediction is from the actual target (e.g., Target 1.0 vs. Prediction 0.48).

>**3.Backpropagation** (The "Blame" Phase): Using the Chain Rule, you calculate "deltas" ($\delta$). This identifies which neurons and weights were most responsible for the error.

>**4.Weight Update:** Using Gradient Descent, we adjust the weights in the direction that reduces error. Small weights change based on the learning rate, the input strength, and the calculated "blame."

By re-running the forward pass after one update, we proved that the error decreases, demonstrating that the network has "learned" from its mistake.
